# H3 Test

---

A Jupyter notebook to test automating the downloading and display of H3 hexagon population data.

Learning resources I've used:

* https://realpython.com/python-download-file-from-url/

### Downloading the files

In [ ]:
from urllib.request import urlretrieve # built in, handles simple requests
import requests # handles more complicated web request/ larger files/ multiple downloads

import pycountry

In [ ]:
# Example URL for ANDORRA
# https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/kontur_population_AD_20231101.gpkg.gz

# Therefore base URL is
base_url = (
    "https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/"
    "kontur_population_{country_alpha_2}_20231101.gpkg.gz"
    )

selected_country = "Italy" # example selected country

selected_country_alpha_2 = pycountry.countries.get(name=selected_country).alpha_2
print(selected_country_alpha_2)

target_url = base_url.format(country_alpha_2=selected_country_alpha_2)
print(target_url)

In [ ]:
response = requests.get(target_url) # simple way of downloading

In [ ]:
# Can do the same with data streaming
response = requests.get(target_url, stream=True)

In [ ]:
response.headers

In [ ]:
# 
with requests.get(target_url, stream=True) as response:
    with open(f"H3_zipped_pop_density_maps/{selected_country_alpha_2}_H3_population_density_map.gpkg.gz", mode="wb") as file:
        for chunk in response.iter_content(chunk_size= 10 * 1024):
            file.write(chunk)

This (above) downloads the population density but we need to unzip etc. so lets make a function

In [ ]:
from pycountry import countries
import os
from pathlib import Path

In [ ]:
def download_H3_population_density_zipped(country_name, output_dir="H3_zipped_pop_density_maps", progress_callback= None, overwrite_download = False):
    try:
        selected_country = countries.get(name = country_name)
        selected_country_alpha_2 = selected_country.alpha_2
    except LookupError:
        return False, f"Country, {country_name}, not found"
    
    base_url = (
    "https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/"
    "kontur_population_{country_alpha_2}_20231101.gpkg.gz"
    )

    target_url = base_url.format(country_alpha_2=selected_country_alpha_2)

    output_file = Path(output_dir)/ f"{selected_country_alpha_2}_H3_population_density_map.gpkg.gz"


    # Check if file exists and we shouldn't overwrite
    if not overwrite_download and os.path.exists(output_file):
        return True, f"File already exists at:\n{output_file}"

    try:
        # Use session to enable connection pooling and compression
        with requests.session() as session:

            with session.get(target_url, stream=True) as response:
                response.raise_for_status()
                total_size = int(response.headers.get("Content-Length",0))

                with open(output_file, mode="wb") as file:
                    
                    downloaded = 0
                    for chunk in response.iter_content(chunk_size= 10 * 1024):
                        file.write(chunk)
                        downloaded += len(chunk)
                        if progress_callback and total_size>0:
                            progress = int(100* downloaded/total_size)
                            progress_callback(progress)
    
    except requests.exceptions.RequestException as e:
        # Handle incomplete shit
        if Path.exists(output_file):
            Path.unlink(output_file)
        return False, f"Download failed, {str(e)}"
    except Exception as e:
        return False, f"Error occured: {str(e)}"
    
# Test run
download_H3_population_density_zipped("Italy")


Now need to decompress (used chatgpt as couldnt see a better way to uncompress)


In [ ]:
import gzip, shutil
import geopandas as gpd

output_dir = "H3_zipped_pop_density_maps"

gz_path = Path(output_dir)/ f"{selected_country_alpha_2}_H3_population_density_map.gpkg.gz"

gpkg_path = gz_path.with_suffix("")

# Unzip

with gzip.open(gz_path, 'rb') as f_in, open(gpkg_path, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)

gdf = gpd.read_file(gpkg_path)
gdf.head()

Now need to work out how to plot the H3 hexagons. Have found this online which works on the same Kontur population data sets (https://www.kaggle.com/code/mpwolke/population-density-h3-hexagons-gpkg) 

In [ ]:
import matplotlib as plt

gdf.shape


In [ ]:
gdf.population.value_counts()

In [ ]:
plt.rcParams['figure.figsize']= 10,10
plt.rcParams['legend.loc']='best'

In [ ]:
print(gdf.geometry[4])

In [ ]:
gdf.geometry[4]

### Map type Tests

In [ ]:
gdf.plot(
    column="population",
    legend=True,
)


In [ ]:
# Basic stats
gdf['population'].describe()


In [ ]:
print(gdf.crs)


Static map (quick & pretty) with GeoPandas + Contextily

In [ ]:
import contextily as ctx
import matplotlib.pyplot as plt

plot_col = 'population'  # or 'population'
ax = gdf.plot(
    column=plot_col,
    cmap='viridis',        # try 'plasma', 'magma', 'inferno', 'turbo'
    scheme='quantiles',    # or 'naturalbreaks', 'equalinterval'
    k=7,
    legend=True,
    linewidth=0
)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_axis_off()
plt.tight_layout()
plt.show()

One-liner interactive map (Geopandas .explore)

This uses Folium

In [ ]:
import folium
# gdf.explore()

# this is commented out because its bloody hard to run

# trying this

m = gdf.explore(
    column='population',    # or 'population'
    cmap='viridis',
    tiles='CartoDB positron',
    tooltip=['h3','population'],
    style_kwds=dict(fillOpacity=0.6, weight=0)
)
#m.save("h3_population_explore.html")

Folium (more control: multiple base tiles, custom legends)

In [ ]:
import folium
import branca.colormap as cm
import json

gjson = json.loads(gdf.to_json())
vmin, vmax = gdf['population'].min(), gdf['population'].max()
colormap = cm.linear.Viridis_09.scale(vmin, vmax)

center = gdf.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=9, tiles=None)

# Base layers
folium.TileLayer('cartodbpositron', name='Positron').add_to(m)
folium.TileLayer('stamenterrain', name='Terrain').add_to(m)
folium.TileLayer('openstreetmap', name='OSM').add_to(m)

folium.GeoJson(
    gjson,
    name='H3 population',
    style_function=lambda feat: {
        "fillColor": colormap(feat['properties']['population']),
        "color": None, "fillOpacity": 0.6, "weight": 0
    },
    tooltip=folium.GeoJsonTooltip(fields=['h3','population'])
).add_to(m)

colormap.caption = "Population"
colormap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m.save("h3_population_folium.html")


https://www.youtube.com/watch?v=I9mt37nd2kg

PyDeck / deck.gl (fast, H3 native layer)

Deck.gl has an H3HexagonLayer that consumes the h3 index directly

In [ ]:
import pydeck as pdk
import numpy as np

df = gdf[['h3','population']].copy()
# Convert your metric to RGBA (or pass a column name and let get_fill_color pick it)
z = df['population']
z01 = (z - z.min()) / (z.max() - z.min() + 1e-9)
df['rgba'] = (z01*255).astype(int).map(lambda v: [v, 0, 255 - v, 180])

# Center from bounds
minx, miny, maxx, maxy = gdf.total_bounds
center_lon = (minx + maxx) / 2
center_lat = (miny + maxy) / 2

# (Optional) quick-and-ok zoom guess from lon span
lon_span = max(1e-6, maxx - minx)
zoom = float(np.clip(np.log2(360.0 / lon_span), 2, 12))  # clamp to sensible range

view_state = pdk.ViewState(
    latitude=center_lat,
    longitude=center_lon,
    zoom=zoom,        # try 8–10 if you prefer a fixed zoom
    pitch=0,
    bearing=0
)

layer = pdk.Layer(
    "H3HexagonLayer",
    df,
    get_hexagon="h3",
    get_fill_color="rgba",
    get_line_color=[40,40,40],
    line_width_min_pixels=0.5,
    pickable=True,
    extruded=False,  # set True + get_elevation to make 3D
    opacity=0.8
)

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style="carto-positron",
    tooltip={"text": "H3: {h3}\nPop: {population}"}
)
r.to_html("H3_maps/h3_population_pydeck.html", notebook_display=True)


In [ ]:
r.show()

### Investigating more into pydeck

In [ ]:
import gzip, shutil
import geopandas as gpd

selected_country = "Italy" # example selected country
selected_country_alpha_2 = pycountry.countries.get(name=selected_country).alpha_2

output_dir = "H3_zipped_pop_density_maps"

gz_path = Path(output_dir)/ f"{selected_country_alpha_2}_H3_population_density_map.gpkg.gz"

gpkg_path = gz_path.with_suffix("")

# Unzip

with gzip.open(gz_path, 'rb') as f_in, open(gpkg_path, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)

gdf = gpd.read_file(gpkg_path)
gdf.head()

In [ ]:
import numpy as np
import pandas as pd
import pydeck as pdk

In [ ]:
gdf = gpd.read_file(gpkg_path)
df = gdf[["h3","population"]].copy

In [ ]:
# --- 2) Color + elevation mapping (simple linear ramp) ---
v = gdf["population"].to_numpy()
vmin, vmax = float(np.nanmin(v)), float(np.nanmax(v) if np.nanmax(v) > 0 else 1)

# map value to 0..255 for a purple→pink look (R,G,B,A)
scale = ((v - vmin) / (vmax - vmin + 1e-9) * 255).astype(int)
gdf["rgba"] = [[int(s), 0, 255 - int(s), 180] for s in scale]

# height in meters (adjust to taste)
gdf["elev"] = (v - vmin) / (vmax - vmin + 1e-9) * 1000.0

# --- 3) H3 layer ---
layer = pdk.Layer(
    "H3HexagonLayer",
    gdf,
    pickable=True,
    auto_highlight=True,
    get_hexagon="h3",
    get_fill_color="rgba",
    get_line_color=[0, 0, 0, 60],
    stroked=True,
    extruded=True,
    get_elevation="elev",
    elevation_scale=1.0,
    opacity=0.85,
)

# --- 4) View (IMPORTANT) ---
# Don't do: ViewState(**gdf.total_bounds)  ← this caused your error.
# Instead provide explicit latitude, longitude, zoom.
# If you know a rough center, put it here:
view_state = pdk.ViewState(latitude=51.5, longitude=-0.12, zoom=8.5, pitch=30)

# Optional basemap; set a token once if you want tiles
# pdk.settings.mapbox_api_key = "YOUR_MAPBOX_TOKEN"
# map_style = "mapbox://styles/mapbox/light-v11"
map_style = None  # works without any token

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style=map_style,
    tooltip={"text": "H3: {h3}\nValue: {value}"},
)
